## 2. EDA Silver — Análise de Qualidade de Dados (Bronze → Silver)

### a) Imports e path

In [0]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pyspark.sql.functions as F

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(project_root)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Workspace/Repos/pierinicoelho@gmail.com


### b) Carregando a base Bronze

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import Tables

bronze_table = f"{Tables.SCHEMA}.{Tables.BRONZE_TAXI}"
df_bronze = spark.table(bronze_table)

total = df_bronze.count()
print(f"Tabela: {bronze_table}")
print(f"Total de registros: {total:,}  |  Colunas: {len(df_bronze.columns)}")
display(df_bronze.limit(5))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Tabela: ifood_taxi_case.taxi_bronze
Total de registros: 16,186,386  |  Colunas: 21


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file_path,etl_ingestion_timestamp
2,2023-01-01T00:32:10.000,2023-01-01T00:40:36.000,1,0.97,1,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0,/Volumes/workspace/ifood_taxi_case/volume_bronze_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-27T23:08:29.860Z
2,2023-01-01T00:55:08.000,2023-01-01T01:01:27.000,1,1.1,1,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0,/Volumes/workspace/ifood_taxi_case/volume_bronze_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-27T23:08:29.860Z
2,2023-01-01T00:25:04.000,2023-01-01T00:37:49.000,1,2.51,1,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0,/Volumes/workspace/ifood_taxi_case/volume_bronze_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-27T23:08:29.860Z
1,2023-01-01T00:03:48.000,2023-01-01T00:13:25.000,0,1.9,1,N,138,7,1,12.1,7.25,0.5,0.0,0.0,1.0,20.85,0.0,1.25,/Volumes/workspace/ifood_taxi_case/volume_bronze_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-27T23:08:29.860Z
2,2023-01-01T00:10:29.000,2023-01-01T00:21:19.000,1,1.43,1,N,107,79,1,11.4,1.0,0.5,3.28,0.0,1.0,19.68,2.5,0.0,/Volumes/workspace/ifood_taxi_case/volume_bronze_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-27T23:08:29.860Z


### c) Perfil de NULLs por coluna

Objetivo: identificar quais colunas têm dados null e em qual magnitude. \
Importante entendermos posteriormente, se colunas null tem valores em outras colunas para nao remove-las e perder dados importantes.

In [0]:
%load_ext autoreload
%autoreload 2

null_exprs = [
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in df_bronze.columns
]

df_nulls = df_bronze.agg(*null_exprs).toPandas().T
df_nulls.columns = ["nulls"]
df_nulls = df_nulls.reset_index(
df_nulls = df_nulls.rename(columns={"index": "nome_da_coluna"}) 
df_nulls["pct"] = (df_nulls["nulls"] / total * 100).round(4)
df_nulls = df_nulls[df_nulls["nulls"] > 0].sort_values("nulls", ascending=False)

print("Colunas com NULLs:")
display(df_nulls)
print(f"\nColunas sem nenhum NULL: {len(df_bronze.columns) - len(df_nulls)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Colunas com NULLs:


nome_da_coluna,nulls,pct
passenger_count,428665,2.6483
RatecodeID,428665,2.6483
store_and_fwd_flag,428665,2.6483
congestion_surcharge,428665,2.6483
airport_fee,428665,2.6483



Colunas sem nenhum NULL: 16


### d) Correlação NULLs × VendorID

As colunas com null apresentam exatamente a mesma contagem — indicando que são os **mesmos registros** com múltiplos campos ausentes. \
A investigação por VendorID permite entender se a distribuiçao dos nulls é uniforme ou concentrada.

In [0]:
%load_ext autoreload
%autoreload 2

df_null_rows = df_bronze.filter(F.col("passenger_count").isNull())

print(f"Registros com passenger_count NULL: {df_null_rows.count():,}")
print("\nDistribuição por VendorID:")
display(
    df_null_rows.groupBy("VendorID")
    .count()
    .withColumn("pct", F.round(F.col("count") / total * 100, 4))
    .orderBy("VendorID")
)

null_cols = ["passenger_count", "RatecodeID", "store_and_fwd_flag", "congestion_surcharge", "airport_fee"]
simultaneous_nulls = df_null_rows.filter(
    F.col("RatecodeID").isNull() &
    F.col("store_and_fwd_flag").isNull() &
    F.col("congestion_surcharge").isNull() &
    F.col("airport_fee").isNull()
).count()
print(f"\nRegistros com TODAS as 5 colunas nulas simultaneamente: {simultaneous_nulls:,}")
print("Conclusão: NULLs são sistêmicos — mesma batch de registros com múltiplos campos ausentes por falha do fornecedor.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Registros com passenger_count NULL: 428,665

Distribuição por VendorID:


VendorID,count,pct
1,112359,0.6942
2,312323,1.9295
6,3983,0.0246



Registros com TODAS as 5 colunas nulas simultaneamente: 428,665
Conclusão: NULLs são sistêmicos — mesma batch de registros com múltiplos campos ausentes por falha do fornecedor.


### e) Anomalias estruturais

Anomalias estruturais são impossibilidades físicas que independem de regra de negócio.
Único caso de remoção na Silver:
- `tpep_dropoff_datetime < tpep_pickup_datetime`: corrida no tempo — fisicamente impossível

**`total_amount < 0` não é removido na Silver.** A análise da distribuição por `payment_type` revela que
a maioria desses registros corresponde a `payment_type=4` (Dispute), onde o valor negativo representa um
chargeback — dinheiro devolvido ao passageiro. \
Isso é um evento de negócio legítimo, não uma corrupção de dado. \
##### ** A decisão de filtrar fica para a camada Gold, conforme o contexto analítico. **

In [0]:
%load_ext autoreload
%autoreload 2

# Desc. tipos de pagamento
payment_desc_expr = (
    F.when(F.col("payment_type") == 0, "0 - Flex Fare trip")
    .when(F.col("payment_type") == 1, "1 - Credit card")
    .when(F.col("payment_type") == 2, "2 - Cash")
    .when(F.col("payment_type") == 3, "3 - No charge")
    .when(F.col("payment_type") == 4, "4 - Dispute")
    .when(F.col("payment_type") == 5, "5 - Unknown")
    .when(F.col("payment_type") == 6, "6 - Voided trip")
    .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("payment_type").cast("string")))
)

# Chegada < Partida
df_time_travel = df_bronze.filter(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"))
time_travel_count = df_time_travel.count()
print(f"Viagens no tempo (dropoff < pickup): {time_travel_count:,}  ({time_travel_count/total*100:.4f}%)")

display(
    df_time_travel
    .withColumn("payment_type_desc", payment_desc_expr)
    .select("VendorID", "total_amount", "payment_type_desc")
    .groupby("VendorID")
    .count()
    .withColumn("pct", F.round(F.col("count") / time_travel_count * 100, 2))
    .orderBy(F.col("count").desc())
)

# Ammounts < 0
df_neg_total = df_bronze.filter(F.col("total_amount") < 0)
neg_total_count = df_neg_total.count()
print(f"\ntotal_amount < 0: {neg_total_count:,}  ({neg_total_count/total*100:.4f}%)")

print("\nDistribuição por payment_type (total_amount < 0):")
display(
    df_neg_total
    .withColumn("payment_type_desc", payment_desc_expr)
    .groupBy("payment_type_desc")
    .count()
    .withColumn("pct", F.round(F.col("count") / neg_total_count * 100, 2))
    .orderBy(F.col("count").desc())
)

print(f"Min total_amount: {df_bronze.agg(F.min('total_amount')).collect()[0][0]:.2f}")
print(f"Max total_amount: {df_bronze.agg(F.max('total_amount')).collect()[0][0]:.2f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Viagens no tempo (dropoff < pickup): 795  (0.0049%)


VendorID,count,pct
6,746,93.84
1,49,6.16



total_amount < 0: 141,407  (0.8736%)

Distribuição por payment_type (total_amount < 0):


payment_type_desc,count,pct
4 - Dispute,82930,58.65
2 - Cash,32949,23.3
3 - No charge,25240,17.85
0 - Flex Fare trip,172,0.12
1 - Credit card,116,0.08


Min total_amount: -982.95
Max total_amount: 6304.90


### f) Datas fora do range Jan–Mai 2023

A ingestão cobriu Janeiro a Maio de 2023. Registros com `tpep_pickup_datetime` fora desse intervalo indicam
dados corrompidos na origem (e.g., erros de digitação, transmissões retroativas). \
##### ** Serão removidos na Silver. **

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import TARGET_YEARS, TARGET_MONTHS

date_min = f"{TARGET_YEARS[0]}-{TARGET_MONTHS[0]}-01"
date_max = f"{TARGET_YEARS[0]}-{int(TARGET_MONTHS[-1])+1:02d}-01"

df_out_of_range = df_bronze.filter(
    (F.col("tpep_pickup_datetime") < date_min) |
    (F.col("tpep_pickup_datetime") >= date_max)
)
out_count = df_out_of_range.count()

print(f"Range esperado: {date_min} a {date_max}")
print(f"Registros fora do range: {out_count:,}  ({out_count/total*100:.4f}%)")

print("\nExtremos de pickup fora do range:")
display(
    df_out_of_range
    .select(
            F.col("tpep_pickup_datetime").cast("date"),
            F.col("tpep_dropoff_datetime").cast("date"),
             "VendorID",
             "total_amount"
             )
    .groupBy(F.col("tpep_pickup_datetime").cast("date"))
    .count()
    .orderBy("tpep_pickup_datetime")
    .limit(10)
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Range esperado: 2023-01-01 a 2023-06-01
Registros fora do range: 104  (0.0006%)

Extremos de pickup fora do range:


tpep_pickup_datetime,count
2001-01-01,6
2002-12-31,4
2003-01-01,5
2008-12-31,13
2009-01-01,2
2014-11-19,1
2022-10-24,4
2022-10-25,7
2022-12-31,25
2023-06-01,16


### g) Duplicatas

Chave natural de unicidade: `(VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID)`. \
Duplicatas podem ocorrer por re-transmissão de registros store-and-forward ou reprocessamentos na origem.
##### ** A Silver manterá apenas a primeira ocorrência (`row_number = 1`). **

In [0]:
%load_ext autoreload
%autoreload 2

from pyspark.sql import Window

DEDUP_KEY = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"]

window = Window.partitionBy(*DEDUP_KEY).orderBy(F.col("etl_ingestion_timestamp"))

df_with_rn = df_bronze.withColumn("_rn", F.row_number().over(window))
dup_count = df_with_rn.filter(F.col("_rn") > 1).count()

print(f"Registros duplicados a remover: {dup_count:,}  ({dup_count/total*100:.4f}%)")
print(f"Registros únicos pós-dedup:     {total - dup_count:,}")

# Exemplo de grupo com duplicata
print("\nExemplo de registros duplicados (mesmo grupo):")
display(
    df_with_rn.filter(F.col("_rn") > 1)
    .select(*DEDUP_KEY, "_rn", "total_amount", "source_file_path")
    .limit(5)
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Registros duplicados a remover: 125,122  (0.7730%)
Registros únicos pós-dedup:     16,061,264

Exemplo de registros duplicados (mesmo grupo):


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,_rn,total_amount,source_file_path
2,2023-01-01T01:02:53.000,2023-01-01T01:50:18.000,132,181,2,68.35,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet
2,2023-01-01T01:16:09.000,2023-01-01T01:20:53.000,141,263,2,12.2,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet
2,2023-01-01T01:17:11.000,2023-01-01T01:35:18.000,141,226,2,27.6,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet
2,2023-01-01T01:38:10.000,2023-01-01T01:48:51.000,265,265,2,101.0,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet
2,2023-01-01T02:21:13.000,2023-01-01T02:48:02.000,161,223,2,32.5,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet


### h) Dados de negócio — mantidos na Silver

Os casos abaixo apresentam valores atípicos mas **não são impossibilidades estruturais**. \
A Silver preserva esses registros; a decisão de filtragem é da camada Gold, com base em contexto analítico.

- `passenger_count == 0`: pode indicar reposicionamento do veículo ou falha de sensor
- `trip_distance == 0`: pode indicar corrida cancelada ou viagem intra-zona
- `total_amount < 0`: chargebacks de disputas (`payment_type=4`) são eventos de negócio legítimos
- NULLs nas 5 colunas sistêmicas: o registro ainda possui âncora temporal e valor financeiro válidos

In [0]:
%load_ext autoreload
%autoreload 2

zero_pass   = df_bronze.filter(F.col("passenger_count") == 0).count()
zero_dist   = df_bronze.filter(F.col("trip_distance") == 0).count()
null_batch  = df_bronze.filter(F.col("passenger_count").isNull()).count()
neg_total   = df_bronze.filter(F.col("total_amount") < 0).count()

print(f"passenger_count == 0 : {zero_pass:,}  ({zero_pass/total*100:.4f}%)  → mantido na Silver")
print(f"trip_distance == 0   : {zero_dist:,}  ({zero_dist/total*100:.4f}%)  → mantido na Silver")
print(f"total_amount < 0     : {neg_total:,}  ({neg_total/total*100:.4f}%)  → mantido na Silver (chargeback)")
print(f"NULLs sistêmicos     : {null_batch:,}  ({null_batch/total*100:.4f}%)  → mantido na Silver")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
passenger_count == 0 : 273,481  (1.6896%)  → mantido na Silver
trip_distance == 0   : 222,617  (1.3753%)  → mantido na Silver
total_amount < 0     : 141,407  (0.8736%)  → mantido na Silver (chargeback)
NULLs sistêmicos     : 428,665  (2.6483%)  → mantido na Silver


### i) Sumário — regras de cleansing da camada Silver

| Regra | Critério | Volume | % | Decisão |
|---|---|---|---|---|
| Dedup | Chave composta duplicada | ~125.122 | ~0,77% | **Remove** |
| Viagem no tempo | `dropoff < pickup` | ~795 | ~0,005% | **Remove** |
| Data fora do range | `pickup` fora de Jan–Mai 2023 | ~104 | ~0,001% | **Remove** |
| NULLs sistêmicos | 5 colunas simultaneamente nulas | ~428.665 | ~2,65% | **Mantém** |
| Total negativo | `total_amount < 0` | ~141.407 | ~0,87% | **Mantém** (chargeback) |
| Passageiro zero | `passenger_count == 0` | ~273.481 | ~1,69% | **Mantém** |
| Distância zero | `trip_distance == 0` | ~222.617 | ~1,38% | **Mantém** |

**Total estimado removido na Silver**: ~126.021 registros (~0,78%)  
**Base Silver estimada**: ~16.060.365 registros

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import TARGET_YEARS, TARGET_MONTHS
from pyspark.sql import Window

DEDUP_KEY = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"]
date_min  = f"{TARGET_YEARS[0]}-{TARGET_MONTHS[0]}-01"
date_max  = f"{TARGET_YEARS[0]}-{int(TARGET_MONTHS[-1])+1:02d}-01"

window = Window.partitionBy(*DEDUP_KEY).orderBy(F.col("etl_ingestion_timestamp"))

df_silver_preview = (
    df_bronze
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1).drop("_rn")
    .filter(F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime"))
    .filter(
        (F.col("tpep_pickup_datetime") >= date_min) &
        (F.col("tpep_pickup_datetime") < date_max)
    )
)

silver_count = df_silver_preview.count()
removed = total - silver_count

print(f"Bronze original : {total:,}")
print(f"Silver estimado : {silver_count:,}")
print(f"Removidos       : {removed:,}  ({removed/total*100:.4f}%)")
print("\nPré-visualização Silver:")
display(df_silver_preview.limit(5))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Bronze original : 16,186,386
Silver estimado : 16,060,365
Removidos       : 126,021  (0.7786%)

Pré-visualização Silver:


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,source_file_path,etl_ingestion_timestamp
1,2023-01-01T00:06:39.000,2023-01-01T00:26:03.000,1,4.7,1,N,232,263,1,21.2,3.5,0.5,5.0,0.0,1.0,31.2,2.5,0.0,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-28T01:00:26.460Z
1,2023-01-01T00:07:08.000,2023-01-01T00:20:41.000,1,3.9,1,Y,148,140,1,17.7,3.5,0.5,6.8,0.0,1.0,29.5,2.5,0.0,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-28T01:00:26.460Z
1,2023-01-01T00:07:19.000,2023-01-01T00:45:31.000,2,7.8,1,N,249,223,1,37.3,3.5,0.5,8.45,0.0,1.0,50.75,2.5,0.0,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-28T01:00:26.460Z
1,2023-01-01T00:08:33.000,2023-01-01T00:10:20.000,4,0.3,1,N,148,79,1,3.7,3.5,0.5,1.7,0.0,1.0,10.4,2.5,0.0,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-28T01:00:26.460Z
1,2023-01-01T00:09:36.000,2023-01-01T00:09:53.000,1,0.0,5,N,265,265,1,90.0,0.0,0.0,20.0,0.0,1.0,111.0,0.0,0.0,/Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet,2026-06-28T01:00:26.460Z
